In [0]:
from datetime import datetime
from pyspark.sql import functions as F

s = spark.table("workspace.m5.silver_sales_daily")
total = s.count()

checks = [
    ("no_negative_units",     s.filter("units < 0").count() == 0),
    ("no_null_dates",         s.filter("date IS NULL").count() == 0),
    ("unique_id_date",        s.groupBy("id", "date").count().filter("count > 1").count() == 0),
    ("price_coverage_over_95pct", s.filter("sell_price IS NOT NULL").count() / total > 0.95),
    ("ten_stores",            s.select("store_id").distinct().count() == 10),
]

results = spark.createDataFrame(
    [(datetime.now(), name, bool(ok)) for name, ok in checks],
    "run_ts timestamp, check_name string, passed boolean")
results.write.mode("append").saveAsTable("workspace.m5.dq_results")
display(results)

In [0]:
failed = [name for name, ok in checks if not ok]
assert not failed, f"Data quality failed: {failed}. Stopping the pipeline."
print("All data quality checks passed")